## Names (Classification)

In [1]:
import os
from glob import glob

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from unidecode import unidecode


In [2]:
root_dir="/Users/mac/Desktop/Machine Learning/DL/DB/Names2"
file_names=glob(os.path.join(root_dir, "*.txt"))

print(f"Found {len(file_names)} language files")

Found 18 language files


## Labels

In [7]:
unique_labels=sorted(
    os.path.splitext(os.path.basename(file))[0]
    for file in file_names
)

idx_to_label={
    idx:label
    for idx, label in enumerate(unique_labels)
}

label_to_idx={
    label: idx
    for idx, label in idx_to_label.items()
}

print("Unique Labels:", unique_labels)
print("Id to Label:", idx_to_label)
print("Label to id:", label_to_idx)

Unique Labels: ['Arabic', 'Chinese', 'Czech', 'Dutch', 'English', 'French', 'German', 'Greek', 'Irish', 'Italian', 'Japanese', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Scottish', 'Spanish', 'Vietnamese']
Id to Label: {0: 'Arabic', 1: 'Chinese', 2: 'Czech', 3: 'Dutch', 4: 'English', 5: 'French', 6: 'German', 7: 'Greek', 8: 'Irish', 9: 'Italian', 10: 'Japanese', 11: 'Korean', 12: 'Polish', 13: 'Portuguese', 14: 'Russian', 15: 'Scottish', 16: 'Spanish', 17: 'Vietnamese'}
Label to id: {'Arabic': 0, 'Chinese': 1, 'Czech': 2, 'Dutch': 3, 'English': 4, 'French': 5, 'German': 6, 'Greek': 7, 'Irish': 8, 'Italian': 9, 'Japanese': 10, 'Korean': 11, 'Polish': 12, 'Portuguese': 13, 'Russian': 14, 'Scottish': 15, 'Spanish': 16, 'Vietnamese': 17}


In [4]:
def clean_name(name):
    
    # lowercase
    name=name.lower()
    
    # unicode -> ascii
    name=unidecode(name)
    
    # remove unwanted example
    if name=="To The First Page":
        return None
    
    # remove unwanted characters
    for ch in [",", ":", "/", "\\", "\xa0", "1", "(", ")", "[", "]", '"']:
        name=name.replace(ch, "")
        
    # convert hyphen to space
    name=name.replace("-", "")
    
    # remove duplicate spaces
    naem="".join(name.split())
    
    
    return name

## Read Dataset

In [5]:
X_names=[]
Y_labels=[]

for file in file_names:
    language=os.path.splitext(
        os.path.basename(file)
    )[0]
    
    with open(file, encoding='utf-8') as f:
        
        for line in f:
            name=clean_name(line.strip())
            
            if name is None:
                continue
            
            if len(name)==0:
                continue
            
            X_names.append(name)
            Y_labels.append(language)
            
print(len(X_names))

20074


## Vocabulary

In [6]:
pad_token="."
pad_id=0

unique_chars=[pad_token]+sorted(
    set("".join(X_names))
)

idx_to_char={
    idx: ch
    for idx, ch in enumerate(unique_chars)
}

char_to_idx={
    ch: idx
    for idx, ch in idx_to_char.items()
}

print("Unique chars:",unique_chars)

print("Id to char:", idx_to_char)

print("Char to id:", char_to_idx)


Unique chars: ['.', ' ', "'", 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Id to char: {0: '.', 1: ' ', 2: "'", 3: 'a', 4: 'b', 5: 'c', 6: 'd', 7: 'e', 8: 'f', 9: 'g', 10: 'h', 11: 'i', 12: 'j', 13: 'k', 14: 'l', 15: 'm', 16: 'n', 17: 'o', 18: 'p', 19: 'q', 20: 'r', 21: 's', 22: 't', 23: 'u', 24: 'v', 25: 'w', 26: 'x', 27: 'y', 28: 'z'}
Char to id: {'.': 0, ' ': 1, "'": 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28}


## Encode / Decode

In [8]:
def encode(name):
    return [
        char_to_idx[ch]
        for ch in name
    ]
    
def decode(ids):
    
    return "".join(idx_to_char[i]
                   for i in ids
    )
    
    
Y=[
    label_to_idx[label]
    for label in Y_labels
]

X=[
    encode(name)
    for name in X_names
]

## Train / Test Split

In [9]:
X_train, X_test, y_train, y_test=train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

## Dataset Class

In [10]:
class NamesDataset(Dataset):
    def __init__(self, X, Y):
        self.X=X
        self.Y=Y
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_set=NamesDataset(X_train, y_train)
test_set=NamesDataset(X_test, y_test)

## Collate Function

In [11]:
max_name_len=20

def collate_fn(batch):
    x, y=zip(*batch)
    
    max_len=min(
        max(len(name) for name in x),
        max_name_len
    )
    
    x_padded=torch.full(
        (len(x), max_len),
        pad_id, dtype=torch.long
    )
    
    for i, name in enumerate(x):
        name=name[:max_len]
        
        x_padded[i, :len(name)]=torch.tensor(
            name, dtype=torch.long
        )
        
    y=torch.tensor(
        y, dtype=torch.long
    )
    
    return x_padded, y

## Dataset Loader

In [12]:
train_loader=DataLoader(
    train_set,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader=DataLoader(
    test_set,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [14]:
X, Y=next(iter(train_loader))

print("X:", X)
print("Y:", Y)

X: tensor([[18,  7, 14, 14,  7,  9, 20, 11, 16, 17,  0,  0],
        [14, 23, 18, 17, 24,  0,  0,  0,  0,  0,  0,  0],
        [11, 21, 13,  7, 16,  6,  7, 20, 17, 24,  0,  0],
        [21, 11, 22, 22,  3,  0,  0,  0,  0,  0,  0,  0],
        [12,  3,  9, 27,  3,  0,  0,  0,  0,  0,  0,  0],
        [ 5, 20, 17,  5,  7, 22, 22, 11,  0,  0,  0,  0],
        [24, 27, 10, 17,  6, 22, 21,  7, 24,  0,  0,  0],
        [ 7, 24,  7, 16, 22, 17, 24,  0,  0,  0,  0,  0],
        [27,  3,  5, 10,  7, 24, 21, 13, 27,  0,  0,  0],
        [10,  3, 22, 11, 16,  0,  0,  0,  0,  0,  0,  0],
        [13, 17, 16,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [13, 17, 23, 20, 11,  0,  0,  0,  0,  0,  0,  0],
        [14,  3, 22, 10,  7, 27,  0,  0,  0,  0,  0,  0],
        [13, 17, 20, 27,  5,  3, 16,  0,  0,  0,  0,  0],
        [ 5, 10,  3, 11, 15,  0,  0,  0,  0,  0,  0,  0],
        [23, 18, 21,  6,  7, 14, 14,  0,  0,  0,  0,  0],
        [12,  3, 13, 17, 24, 14,  7, 16, 13, 17,  0,  0],
        [ 3

## Model (CNN)

In [15]:
import torch
import torch.nn as nn

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        
        # character embeding
        self.embedding=nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim=embedding_dim,
            padding_idx=pad_id
        )
        
        # Feature Extraction
        self.features=nn.Sequential(
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            
            nn.BatchNorm1d(64),
            
            nn.ReLU(),
            
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            
            nn.BatchNorm1d(128),
            
            nn.ReLU(),
            
            nn.AdaptiveAvgPool1d(1)
        )
        
        ## Classifier
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
        
    def forward(self, x):
        # x (batch_size, sequence_length)
        x=self.embedding(x)
        
        # (batch, seq, embedding)
        x=x.permute(0, 2, 1)
        
        # (batch, embedding, seq)
        x=self.features(x)
        
        # (batch, 128, 1)
        x=self.classifier(x)
        
        # (batch, num_classes)
        
        return x

## create model

In [16]:
VOCAB_SIZE=len(char_to_idx)
NUM_CLASSES=len(label_to_idx)

model=CNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=64, 
    num_classes=NUM_CLASSES  
)

print(model)

CNNClassifier(
  (embedding): Embedding(29, 64, padding_idx=0)
  (features): Sequential(
    (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.3, inplace=False)
    (5): Linear(in_features=64, out_features=18, bias=True)
  )
)


In [17]:
## Test the model

X, Y=next(iter(train_loader))

print(X.shape)

output=model(X)

print(output.shape)

torch.Size([64, 14])
torch.Size([64, 18])


In [18]:
### count parameters

total_params=sum(p.numel() for p in model.parameters())
trainable_params=sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

Total parameters     : 48,722
Trainable parameters : 48,722


## Training Section

In [19]:
## Device
device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

model=model.to(device)

cpu


In [20]:
## Loss Function

criterion=nn.CrossEntropyLoss()

In [21]:
## Optimizer

optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [22]:
## Accuracy Function

def accuracy(predictions, labels):
    predicted=predictions.argmax(dim=1)
    correct=(predicted==labels).sum().item()
    
    return correct/ len(labels)


## Train Function

In [23]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    
    running_loss=0
    running_acc=0
    
    for X, Y in loader:
        X=X.to(device)
        Y=Y.to(device)
        
        optimizer.zero_grad()
        
        outputs=model(X)
        
        loss=criterion(outputs, Y)
        
        loss.backward()
        
        optimizer.step()
        
        running_loss+=loss.item()
        
        running_acc+=accuracy(outputs, Y)
        
    epoch_loss=running_loss/len(loader)
    epoch_acc=running_acc/len(loader)
    
    return epoch_loss, epoch_acc

## Validation Function

In [24]:
def evaluate(model, loader, criterion, device):
    model.eval()
    
    running_loss=0
    running_acc=0
    
    with torch.no_grad():
        for X, Y in loader:
            X=X.to(device)
            Y=Y.to(device)
            
            outputs=model(X)
            
            loss=criterion(outputs, Y)
            
            running_loss+=loss.item()
            running_acc+=accuracy(outputs, Y)
            
        epoch_loss=running_loss/len(loader)
        epoch_acc=running_acc/len(loader)
        
        return epoch_loss, epoch_acc

## Training Loop

In [25]:
train_losses = []
train_accuracies = []

test_losses = []
test_accuracies = []

epochs=20

best_accuracy=0
for epoch in range(epochs):
    
    train_loss, train_acc=train(
        model,
        train_loader, 
        criterion, 
        optimizer,
        device)
    
    test_loss, test_acc=evaluate(
        model,
        test_loader,
        criterion,
        device)
    
    
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    if test_acc> best_accuracy:
        best_accuracy=test_acc
        
        torch.save(
            model.state_dict(),
            "best_model.pth"
        )
        
        
    print(
        f"Epoch {epoch+1:02}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f}"
    )
    

Epoch 01/20 | Train Loss: 1.4160 | Train Acc: 0.6043 | Test Loss: 1.0755 | Test Acc: 0.7044
Epoch 02/20 | Train Loss: 1.0258 | Train Acc: 0.7169 | Test Loss: 0.8871 | Test Acc: 0.7458
Epoch 03/20 | Train Loss: 0.8941 | Train Acc: 0.7475 | Test Loss: 0.8202 | Test Acc: 0.7619
Epoch 04/20 | Train Loss: 0.8118 | Train Acc: 0.7683 | Test Loss: 0.7399 | Test Acc: 0.7812
Epoch 05/20 | Train Loss: 0.7631 | Train Acc: 0.7805 | Test Loss: 0.7318 | Test Acc: 0.7775
Epoch 06/20 | Train Loss: 0.7232 | Train Acc: 0.7875 | Test Loss: 0.6842 | Test Acc: 0.7898
Epoch 07/20 | Train Loss: 0.7035 | Train Acc: 0.7936 | Test Loss: 0.6657 | Test Acc: 0.7967
Epoch 08/20 | Train Loss: 0.6681 | Train Acc: 0.8026 | Test Loss: 0.6591 | Test Acc: 0.8004
Epoch 09/20 | Train Loss: 0.6609 | Train Acc: 0.8012 | Test Loss: 0.6538 | Test Acc: 0.8026
Epoch 10/20 | Train Loss: 0.6344 | Train Acc: 0.8079 | Test Loss: 0.6334 | Test Acc: 0.8100
Epoch 11/20 | Train Loss: 0.6202 | Train Acc: 0.8143 | Test Loss: 0.6134 | Test 

In [26]:
import pickle

# Save character vocabulary
with open("char2idx.pkl", "wb") as f:
    pickle.dump(char_to_idx, f)

# Save label mappings
with open("idx2label.pkl", "wb") as f:
    pickle.dump(idx_to_label, f)

with open("label2idx.pkl", "wb") as f:
    pickle.dump(label_to_idx, f)

## load the best model

In [27]:
model=CNNClassifier(
    vocab_size=len(char_to_idx),
    embedding_dim=64,
    num_classes=len(label_to_idx)
)

model.load_state_dict(torch.load("best_model.pth", map_location=device))

model.to(device)
model.eval()

print("Model loaded successfully!")

Model loaded successfully!


## Prediction Function

In [ ]:
def predict(name):
    
    # Clean the name
    name=clean_name(name)
    
    # Encode characters
    encoded=[
        char_to_idx.get(ch, char_to_idx['.'])
        for ch in name
    ]
    
    
    # Convert to tensor
    x=torch.tensor(encoded, dtype=torch.long).unsqueeze(0)
    
    x=x.to(device)
    
    with torch.no_grad():
        
        output=model(x)
        
        prediction=output.argmax(dim=1).item()
        
    return idx_to_label[prediction]

In [38]:
print(predict("Ivanov"))
print(predict("Ahmed"))
print(predict("Yamamoto"))
print(predict("Smith"))
print(predict("Garcia"))

ivanov Encoded [11, 24, 3, 16, 17, 24]
Russian
ahmed Encoded [3, 10, 15, 7, 6]
Russian
yamamoto Encoded [27, 3, 15, 3, 15, 17, 22, 17]
Japanese
smith Encoded [21, 15, 11, 22, 10]
English
garcia Encoded [9, 3, 20, 5, 11, 3]
Italian


In [36]:
print(predict("Temurbek"))
print(predict("wang"))

Russian
English
